In [1]:
import pandas as pd

df = pd.read_csv("C:/bank-grade-data-pipeline/data/processed/clean_loans.csv", encoding="latin1", low_memory=False)


In [2]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])]

In [3]:
df['default'] = df['loan_status'].map({'Charged Off':1, 'Default':1, 'Fully Paid':0})

In [4]:
for i in df.columns:
    print(i)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
b

In [10]:
features = [
    'loan_amnt', 'term', 'int_rate', 'installment',
    'annual_inc', 'dti', 'emp_length', 'home_ownership', 'purpose',
    'revol_util', 'open_acc', 'delinq_2yrs', 'total_acc'
]


df_model = df[features + ['default']]


In [11]:
from sklearn.model_selection import train_test_split

X = df[features]

y = df['default']

X_train, x_test, Y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [12]:
df_model['emp_length'] = df_model['emp_length'].fillna('Unknown')
df_model['annual_inc'] = df_model['annual_inc'].fillna(df_model['annual_inc'].median())
df_model['dti'] = df_model['dti'].fillna(df_model['dti'].median())


C:\Users\yukta\AppData\Local\Temp\ipykernel_23844\1107692028.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model['emp_length'] = df_model['emp_length'].fillna('Unknown')
C:\Users\yukta\AppData\Local\Temp\ipykernel_23844\1107692028.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model['annual_inc'] = df_model['annual_inc'].fillna(df_model['annual_inc'].median())
C:\Users\yukta\AppData\Local\Temp\ipykernel_23844\1107692028.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of

In [13]:
pip install optbinning


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: optbinning in c:\users\yukta\miniconda3\lib\site-packages (0.21.0)




[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from optbinning import OptimalBinning

woe_mappings = {}
X_train_woe = pd.DataFrame()
X_test_woe = pd.DataFrame()

for col in X_train.columns:
    # Determine type
    if X_train[col].dtype.name in ['object', 'category']:
        dtype = 'categorical'
    else:
        dtype = 'numerical'

    optb = OptimalBinning(
        name=col,
        dtype=dtype,
        max_n_bins=5,
        monotonic_trend='auto' if dtype == 'numerical' else None,
        solver='cp'
    )

    optb.fit(X_train[col], Y_train)

    # Transform to WoE
    X_train_woe[col] = optb.transform(X_train[col], metric='woe')
    X_test_woe[col] = optb.transform(x_test[col], metric='woe')

    # Store mapping for production
    woe_mappings[col] = optb


In [ ]:

bin_table = woe_mappings['revol_util'].binning_table.build()
import numpy as np

def compute_iv(bin_table):
    # Avoid division by zero
    bin_table = bin_table.copy()
    bin_table['dist_event'] = bin_table['Event'] / bin_table['Event'].sum()
    bin_table['dist_non_event'] = bin_table['Non-event'] / bin_table['Non-event'].sum()
    # Avoid log(0)
    bin_table['woe'] = np.log(
        (bin_table['dist_non_event'].replace(0, 0.0001)) /
        (bin_table['dist_event'].replace(0, 0.0001))
    )
    bin_table['iv'] = (bin_table['dist_non_event'] - bin_table['dist_event']) * bin_table['woe']
    return bin_table['iv'].sum()
iv_dict = {}

for col, optb in woe_mappings.items():
    bin_table = optb.binning_table.build()
    iv_dict[col] = compute_iv(bin_table)

iv_df = pd.DataFrame.from_dict(iv_dict, orient='index', columns=['IV']).sort_values(by='IV', ascending=False)
iv_df


,IV
int_rate,0.239332
term,0.067885
dti,0.037048
home_ownership,0.023336
installment,0.017930
revol_util,0.017663
annual_inc,0.017422
loan_amnt,0.015349
purpose,0.007732
emp_length,0.007598


In [25]:
selected_features = ['int_rate', 'term', 'dti', 'home_ownership',"annual_inc",
"loan_amnt" ]

In [27]:
X_train_woe_sel = pd.DataFrame()
X_test_woe_sel = pd.DataFrame()

for col in selected_features:
    optb = woe_mappings[col]
    X_train_woe_sel[col] = optb.transform(X_train[col], metric='woe')
    X_test_woe_sel[col] = optb.transform(x_test[col], metric='woe')


In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_woe_sel, Y_train)

# Predict probabilities
y_pred_proba = lr.predict_proba(X_test_woe_sel)[:, 1]

# Model performance
auc = roc_auc_score(y_test, y_pred_proba)
gini = 2 * auc - 1
print(f"AUC: {auc:.4f}, Gini: {gini:.4f}")


AUC: 0.7042, Gini: 0.4085


In [30]:
coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': lr.coef_[0]
}).sort_values(by='Coefficient', ascending=False)
coef_df


,Feature,Coefficient
1,term,-0.390465
2,dti,-0.490604
4,annual_inc,-0.617336
5,loan_amnt,-0.758524
0,int_rate,-0.833162
3,home_ownership,-0.929037


In [33]:
import pickle

# Save logistic regression model
with open("lr_model.pkl", "wb") as f:
    pickle.dump(lr, f)


In [34]:
# Save the WoE mappings from optbinning
with open("woe_mappings.pkl", "wb") as f:
    pickle.dump(woe_mappings, f)


In [35]:
import os

print(os.listdir('.'))


['.ipynb_checkpoints', 'lr_model.pkl', 'train_model.py', 'Untitled.ipynb', 'validation_script.py', 'woe_mappings.pkl']
